# Misinformation Models  — Opinion Classification
Driver notebook: runs each model, collects predictions, compares metrics.  
Add new models in the **Run models** cell. Train/dev only — test set not touched.
Had help from Claude on implementation.

### Import configuration and packages ###

In [2]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import pandas as pd
import numpy as np
import torch
import cnn_baseline as cnn

#Running this will import FastText vector file, which is stored on HuggingFace and is >4gb. 
from config import DATA_DIR, FASTTEXT_PATH, TARGETS

from preprocess import preprocess
from metrics import compute_metrics, print_confusion_matrix, print_sklearn_report, error_analysis, print_report

DEVICE = torch.device(
    "mps"  if torch.backends.mps.is_available()  else
    "cuda" if torch.cuda.is_available()           else
    "cpu"
)
print(f"Device: {DEVICE}")
print(f"Targets: {TARGETS}")

/Users/rachelledejager/miniforge3/envs/colx_523/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: mps
Targets: ['opinion_label', 'misinformation_label']


In [3]:
# Load data (shared across all models)
train_rows = cnn.load_csv(DATA_DIR / "mis_df_train.csv")
dev_rows   = cnn.load_csv(DATA_DIR / "mis_df_dev.csv")
print(f"Train: {len(train_rows)} | Dev: {len(dev_rows)}")

Train: 600 | Dev: 200


## Run models


In [4]:
# Each model should produce (preds, labels, probs) and store them in `results`.
# Add new models below following the same pattern.
results = {}

#### CNN Baseline

In [5]:
# Run CNN baseline model 

vocab        = cnn.build_vocab(train_rows, preprocess)
embed_matrix = cnn.load_fasttext_vectors(FASTTEXT_PATH, vocab)
train_loader = cnn.make_loader(train_rows, vocab, shuffle=True,  tokenize_fn=preprocess)
dev_loader   = cnn.make_loader(dev_rows,   vocab, shuffle=False, tokenize_fn=preprocess)

# Baseline CNN: one independent model run per target
for target in TARGETS:
    model = cnn.TextCNN(len(vocab), embed_matrix).to(DEVICE)
    model = cnn.train_model(model, train_loader, dev_loader, train_rows, DEVICE,
                            train_targets=[target])
    results[f"TextCNN — {target}"] = cnn.predict(model, dev_loader, DEVICE, target=target)

Loading FastText vectors from /Users/rachelledejager/.cache/huggingface/hub/datasets--COLX523--fasttext-cc-en-300/snapshots/63ba06b23eb770a6fe0f6c17b0de981e43aec6aa/cc.en.300.vec …
  3875/4091 vocab tokens found in FastText vectors (94.7%)
Epoch   1 | loss=0.7467 | avg_dev_f1=0.6976 (opinion_label: 0.6976)
Epoch   2 | loss=0.6497 | avg_dev_f1=0.6928 (opinion_label: 0.6928)
Epoch   3 | loss=0.6010 | avg_dev_f1=0.7093 (opinion_label: 0.7093)
Epoch   4 | loss=0.5371 | avg_dev_f1=0.7014 (opinion_label: 0.7014)
Epoch   5 | loss=0.5012 | avg_dev_f1=0.7177 (opinion_label: 0.7177)
Epoch   6 | loss=0.4437 | avg_dev_f1=0.7172 (opinion_label: 0.7172)
Epoch   7 | loss=0.3909 | avg_dev_f1=0.6900 (opinion_label: 0.6900)
Epoch   8 | loss=0.3446 | avg_dev_f1=0.7118 (opinion_label: 0.7118)
Epoch   9 | loss=0.2789 | avg_dev_f1=0.7261 (opinion_label: 0.7261)
Epoch  10 | loss=0.2521 | avg_dev_f1=0.7159 (opinion_label: 0.7159)
Epoch  11 | loss=0.2244 | avg_dev_f1=0.7081 (opinion_label: 0.7081)
Epoch  12 | 

#### CNN Transfer Learning

In [6]:
# Run CNN transfer learning model - one model jointly trained on both targets

cnn_model    = cnn.TextCNN(len(vocab), embed_matrix).to(DEVICE)
cnn_model    = cnn.train_model(cnn_model, train_loader, dev_loader, train_rows, DEVICE)

for target in TARGETS:
    results[f"TextCNN Transfer — {target}"] = cnn.predict(cnn_model, dev_loader, DEVICE, target=target)

Epoch   1 | loss=1.7759 | avg_dev_f1=0.6759 (opinion_label: 0.6837 | misinformation_label: 0.6682)
Epoch   2 | loss=1.5748 | avg_dev_f1=0.7325 (opinion_label: 0.6933 | misinformation_label: 0.7717)
Epoch   3 | loss=1.3835 | avg_dev_f1=0.7179 (opinion_label: 0.6726 | misinformation_label: 0.7631)
Epoch   4 | loss=1.2139 | avg_dev_f1=0.7420 (opinion_label: 0.6869 | misinformation_label: 0.7971)
Epoch   5 | loss=1.0466 | avg_dev_f1=0.7503 (opinion_label: 0.6846 | misinformation_label: 0.8160)
Epoch   6 | loss=0.9388 | avg_dev_f1=0.7675 (opinion_label: 0.6893 | misinformation_label: 0.8456)
Epoch   7 | loss=0.8396 | avg_dev_f1=0.7682 (opinion_label: 0.6869 | misinformation_label: 0.8496)
Epoch   8 | loss=0.7500 | avg_dev_f1=0.7754 (opinion_label: 0.6976 | misinformation_label: 0.8533)
Epoch   9 | loss=0.6581 | avg_dev_f1=0.7772 (opinion_label: 0.7029 | misinformation_label: 0.8515)
Epoch  10 | loss=0.5871 | avg_dev_f1=0.7609 (opinion_label: 0.6782 | misinformation_label: 0.8435)
Epoch  11 

#### Logistic Regression Baseline

In [7]:
# Run Logistic Regression model — trained separately per target
import logreg_baseline as lr

for target in TARGETS:
    results[f"LogReg — {target}"] = lr.run(train_rows, dev_rows, task=target)


── Logistic Regression  [opinion_label] ──
  Building features (TF-IDF + linguistic)
  Feature matrix: train=(600, 2504), dev=(200, 2504)
  Running GridSearchCV over C
  Best C: 1.0  |  CV macro-F1: 0.7013

── Logistic Regression  [misinformation_label] ──
  Building features (TF-IDF + linguistic)
  Feature matrix: train=(600, 2504), dev=(200, 2504)
  Running GridSearchCV over C
  Best C: 1.0  |  CV macro-F1: 0.8514


#### Logistic Regression Transfer Learning

In [8]:
# Run Logistic Regression transfer model — trained separately per target
import logreg_transfer as lr_transfer

for target in TARGETS:
    results[f"LogReg (+ embeddings) — {target}"] = lr_transfer.run(train_rows, dev_rows, task=target)


── Logistic Regression  [opinion_label] ──
  Building features (TF-IDF, linguistic features, FastText embeddings)
Loading FastText vectors from /Users/rachelledejager/.cache/huggingface/hub/datasets--COLX523--fasttext-cc-en-300/snapshots/63ba06b23eb770a6fe0f6c17b0de981e43aec6aa/cc.en.300.vec …
  3566/4375 vocab tokens found in FastText vectors (81.5%)
  Feature matrix: train=(600, 2804), dev=(200, 2804)
  Running GridSearchCV over C
  Best C: 1.0  |  CV macro-F1: 0.7269

── Logistic Regression  [misinformation_label] ──
  Building features (TF-IDF, linguistic features, FastText embeddings)
Loading FastText vectors from /Users/rachelledejager/.cache/huggingface/hub/datasets--COLX523--fasttext-cc-en-300/snapshots/63ba06b23eb770a6fe0f6c17b0de981e43aec6aa/cc.en.300.vec …
  3566/4375 vocab tokens found in FastText vectors (81.5%)
  Feature matrix: train=(600, 2804), dev=(200, 2804)
  Running GridSearchCV over C
  Best C: 1.0  |  CV macro-F1: 0.8716


#### Results

In [9]:
#Create table to compare models across metrics
rows = []
for name, (preds, labels, probs) in results.items():
    m = compute_metrics(preds, labels, probs)
    rows.append({"Model": name, "Macro F1": m["macro_f1"],
                 "F1 (not-op)": m["f1_class0"], "F1 (opinion)": m["f1_class1"],
                 "F1.5 (recall-weighted)": m["fbeta_class1"],
                 "AUC-ROC": m.get("auc_roc", float("nan"))})

pd.set_option("display.float_format", "{:.4f}".format)
pd.DataFrame(rows).set_index("Model")

,Macro F1,F1 (not-op),F1 (opinion),F1.5 (recall-weighted),AUC-ROC
Model,,,,,
TextCNN — opinion_label,0.7261,0.7589,0.6932,0.7087,0.7930
TextCNN — misinformation_label,0.8773,0.9356,0.8190,0.8221,0.9526
TextCNN Transfer — opinion_label,0.6963,0.7297,0.6629,0.6806,0.7701
TextCNN Transfer — misinformation_label,0.8742,0.9365,0.8119,0.8027,0.9593
LogReg — opinion_label,0.6931,0.7391,0.6471,0.6530,0.7514
LogReg — misinformation_label,0.8784,0.9404,0.8163,0.7975,0.9630
LogReg (+ embeddings) — opinion_label,0.7076,0.7339,0.6813,0.7052,0.7704
LogReg (+ embeddings) — misinformation_label,0.8889,0.9428,0.8350,0.8318,0.9500


In [10]:
#Model details
for name, (preds, labels, probs) in results.items():
    print(f"\n{'='*50}\n{name}\n{'='*50}")
    print_confusion_matrix(preds, labels)
    print()
    print_sklearn_report(preds, labels)
    print()
    error_analysis(dev_rows, preds, labels)


TextCNN — opinion_label
Confusion matrix (rows=true, cols=predicted):
                 pred=0  pred=1
  true=0 (not-op):    85      32
  true=1 (opinion):   22      61

              precision    recall  f1-score   support

 not-opinion       0.79      0.73      0.76       117
     opinion       0.66      0.73      0.69        83

    accuracy                           0.73       200
   macro avg       0.73      0.73      0.73       200
weighted avg       0.74      0.73      0.73       200


False Positives (predicted opinion, actually not) — 5 shown:
  [17] 'Birds in a blizzard. We put out extra sunflower seeds since their usual food sources just got‚Ä¶ https://www.instagram.c'
  [25] "I'm getting used to seeing a layer of ash on my car each morning - and I'm in AB. We've  only seen a red sun this week. "
  [55] "@jihettly @esd2000 good morning! Yes I'm in the middle of the blizzard. I have plenty of food so I'm happy! Lol. Stay wa"
  [59] 'Re: Hurricane Matthew: All of the @ASUTenni

### Simple Ensembling

In [11]:
# Simple ensembling method
import importlib
import simple_ensemble
importlib.reload(simple_ensemble)
from simple_ensemble import soft_vote

ensemble_rows = []
for task in TARGETS:
    ensemble_results = soft_vote([
        results[f"TextCNN — {task}"],
        results[f"LogReg — {task}"],
    ])
    preds, labels, probs = ensemble_results
    m = compute_metrics(preds, labels, probs)
    ensemble_rows.append({"Model": f"Soft Vote Ensemble — {task}", 
                          "Macro F1": m["macro_f1"],
                          "F1 (not-op)": m["f1_class0"], 
                          "F1 (opinion)": m["f1_class1"],
                          "F1.5 (recall-weighted)": m["fbeta_class1"],
                          "AUC-ROC": m["auc_roc"]})

pd.set_option("display.float_format", "{:.4f}".format)
pd.DataFrame(ensemble_rows).set_index("Model")

,Macro F1,F1 (not-op),F1 (opinion),F1.5 (recall-weighted),AUC-ROC
Model,,,,,
Soft Vote Ensemble — opinion_label,0.7097,0.7489,0.6705,0.6811,0.7974
Soft Vote Ensemble — misinformation_label,0.9161,0.9559,0.8762,0.8794,0.9705


### Motivated Ensembling

In [12]:
# Motivational ensembling method
import importlib
import motivated_ensemble
importlib.reload(motivated_ensemble)
from motivated_ensemble import motivated_soft_vote


f1_scores = {
    "opinion_label":        [0.7261, 0.6931],  
    "misinformation_label": [0.8773, 0.8784], 
}

motivated_rows = []
for task in TARGETS:
    preds, labels, probs = motivated_soft_vote(
        model_outputs=[
            results[f"TextCNN — {task}"],
            results[f"LogReg — {task}"],
        ],
        f1_weights=f1_scores[task],
    )
    m = compute_metrics(preds, labels, probs)
    motivated_rows.append({"Model": f"Motivated Ensemble — {task}",
                          "Macro F1": m["macro_f1"],
                           "F1 (not-op)": m["f1_class0"], "F1 (opinion)": m["f1_class1"],
                           "F1.5 (recall-weighted)": m["fbeta_class1"],
                           "AUC-ROC": m["auc_roc"]})

pd.set_option("display.float_format", "{:.4f}".format)
pd.DataFrame(motivated_rows).set_index("Model")


  Threshold sweep (Macro F1 criterion):
   Threshold    Macro F1   F1 (cl.0)   F1 (cl.1)    Accuracy
  ----------------------------------------------------------
        0.30      0.6512      0.6145      0.6878      0.6550
        0.35      0.6840      0.6667      0.7014      0.6850
        0.40      0.7200      0.7228      0.7172      0.7200
        0.45      0.7238      0.7418      0.7059      0.7250 ◄
        0.50      0.7152      0.7522      0.6782      0.7200
        0.55      0.7095      0.7647      0.6543      0.7200
        0.60      0.7013      0.7760      0.6267      0.7200
        0.65      0.6702      0.7774      0.5630      0.7050
        0.70      0.5913      0.7589      0.4237      0.6600

  Selected threshold: 0.45

  Threshold sweep (Macro F1 criterion):
   Threshold    Macro F1   F1 (cl.0)   F1 (cl.1)    Accuracy
  ----------------------------------------------------------
        0.30      0.8756      0.9247      0.8264      0.8950
        0.35      0.9008      0.94

,Macro F1,F1 (not-op),F1 (opinion),F1.5 (recall-weighted),AUC-ROC
Model,,,,,
Motivated Ensemble — opinion_label,0.7238,0.7418,0.7059,0.7377,0.7977
Motivated Ensemble — misinformation_label,0.9180,0.9553,0.8807,0.8966,0.9705
